# Exploratory Data Analysis Practice

**Topic:** pandas, correlation, and group comparison &nbsp;·&nbsp; **Difficulty:** Easy

## The data (created in the next cell)

- `df`: 600 students with columns `hours_studied`, `sleep_hours`, `phone_hours`, `attendance_pct`, and the outcome `exam_score`.
- Exactly one of the habit columns drives most of the score, and the others have only a small effect.
- After computing your answers, practice writing a short Markdown explanation of what you found (graders read those!).

## Your task

Analyze `df` and store your findings in a dict named `answers`:

```python
answers = {
    "strongest_feature": ..., # str: habit column with the largest |Pearson r| vs exam_score
    "strongest_r": ...,       # float: that Pearson correlation (signed)
    "attendance_gap": ...,    # float: mean exam_score of students with attendance_pct >= 90
                               #        minus mean exam_score of the rest
}
```

Habit columns are `hours_studied`, `sleep_hours`, `phone_hours`, `attendance_pct`.

Pearson r is the correlation from the Covariance and correlation section of Expectation and Joint Distributions. `df.corrwith(...)` computes it by default.

## Evaluation

The evaluation cell recomputes the ground truth and checks all three answers (the feature name exactly, and the two floats with tolerance).


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 600

hours = rng.gamma(4.0, 1.2, n).clip(0, 12)
sleep = rng.normal(7.2, 1.1, n).clip(4, 10)
phone = rng.gamma(3.0, 1.0, n).clip(0, 10)
attendance = (100 - rng.gamma(2.0, 4.0, n)).clip(40, 100)

score = (
    38
    + 4.6 * hours
    + 0.9 * sleep
    - 1.4 * phone
    + 0.12 * attendance
    + rng.normal(0, 6.0, n)
).clip(0, 100)

df = pd.DataFrame({
    "hours_studied": hours.round(1),
    "sleep_hours": sleep.round(1),
    "phone_hours": phone.round(1),
    "attendance_pct": attendance.round(0),
    "exam_score": score.round(1),
})
print(f"df shape: {df.shape}")
df.describe().round(2)


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Analyze the DataFrame created above, then create the variable the
# task asks for (the `answers` dict).
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your answers to the truth
# =====================================================================
if "answers" not in dir():
    raise NameError("Define the `answers` dict in the solution cell before running this cell.")

_habits = ["hours_studied", "sleep_hours", "phone_hours", "attendance_pct"]
_corrs = df[_habits].corrwith(df["exam_score"])
_true_feature = _corrs.abs().idxmax()
_true_r = float(_corrs[_true_feature])
_mask = df["attendance_pct"] >= 90
_true_gap = float(df.loc[_mask, "exam_score"].mean() - df.loc[~_mask, "exam_score"].mean())

_ok = 0
print("=" * 52)
_got = str(answers.get("strongest_feature", "MISSING"))
_good = _got == _true_feature; _ok += _good
print(f"  strongest_feature yours={_got:<16} {'CORRECT' if _good else 'WRONG'}")
_got_r = answers.get("strongest_r", None)
_good = _got_r is not None and abs(float(_got_r) - _true_r) < 0.03; _ok += _good
print(f"  strongest_r       yours={_got_r!r:<16} {'CORRECT' if _good else 'WRONG'}")
_got_g = answers.get("attendance_gap", None)
_good = _got_g is not None and abs(float(_got_g) - _true_gap) < 0.5; _ok += _good
print(f"  attendance_gap    yours={_got_g!r:<16} {'CORRECT' if _good else 'WRONG'}")
print("=" * 52)
print(f"Score: {_ok}/3")
print("PASSED. Nice work!" if _ok == 3 else "NOT PASSED yet. Check your pandas calls.")


## Solution

`corrwith` gives all four habit-vs-score correlations at once. Take the largest by absolute value but report the signed r. The attendance gap is a boolean-mask difference of means.

```python
habits = ["hours_studied", "sleep_hours", "phone_hours", "attendance_pct"]
corrs = df[habits].corrwith(df["exam_score"])
strongest = corrs.abs().idxmax()
mask = df["attendance_pct"] >= 90

answers = {
    "strongest_feature": strongest,
    "strongest_r": float(corrs[strongest]),
    "attendance_gap": float(df.loc[mask, "exam_score"].mean() - df.loc[~mask, "exam_score"].mean()),
}
print(answers)
```
